# Day 1: Website Summarizer

Give it a URL and GPT responds with a short, snarky summary of the page.

In [ ]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

## Connect to OpenAI

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found. Add OPENAI_API_KEY to your .env file.")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start with sk-proj-.")
elif api_key.strip() != api_key:
    print("The API key has leading or trailing whitespace.")
else:
    print("API key found and looks good so far!")

openai = OpenAI()

## First call to GPT

In [ ]:
messages = [{"role": "user", "content": "Hello, GPT! This is my first ever message to you! Hi!"}]

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
print(response.choices[0].message.content)

## System and user prompts

The system prompt sets the task and tone; the user prompt is the conversation starter.

In [ ]:
messages = [
    {"role": "system", "content": "You are a Puerto Rican assistant"},
    {"role": "user", "content": "What is 2 + 2?"},
]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
print(response.choices[0].message.content)

## Scrape a website

In [ ]:
ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

## Build the summarizer

In [ ]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""


def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website},
    ]


def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages_for(website))
    return response.choices[0].message.content


def display_summary(url):
    display(Markdown(summarize(url)))

In [ ]:
display_summary("https://edwarddonner.com")

In [ ]:
display_summary("https://home.cegsoft.com/")

In [ ]:
display_summary("https://www.expertradiology.com/")

In [ ]:
display_summary("https://anthropic.com")

## My exercise: a psychologist with a sense of humor

A warm, empathetic psychologist who answers with gentle wit but grounded advice.

In [ ]:
system_prompt = (
    "You are a warm, empathetic psychologist. Respond to patient concerns "
    "using lighthearted humor and gentle wit, but ensure your core advice is deeply caring, "
    "supportive, and grounded in real psychological insight."
)

user_prompt = """
Doc, I keep procrastinating on everything until the absolute last minute.
I have a huge report due tomorrow, and instead of writing it, I just spent three hours
deep-cleaning my microwave and reorganizing my sock drawer. Why do I do this to myself?
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
display(Markdown(response.choices[0].message.content))